# Phase 1 — Test Suite

Systematic tests for every implemented module.

**Sections 1–5** use a tiny CPU mock model (same attribute paths as Qwen, d=16).
They test the damage infrastructure — your Python code — quickly without
downloading the checkpoint. Think of them as unit tests.

**Section 6** loads the real `Qwen/Qwen2.5-3B-Instruct` and runs the same
checks end-to-end. This is the integration test that actually matters.

| Section | What's tested |
|---|---|
| 1 | `disease_state` — all dataclasses and helpers |
| 2 | `damage/noise` — inject + restore |
| 3 | `damage/pruning` — masks + prune + restore |
| 4 | `damage/mechanisms` — apply_all_damage orchestration |
| 5 | `qwen_hooks` — damage_context round-trip + exception safety |
| 6 | Real model — full stack on Qwen/Qwen2.5-3B-Instruct |

In [ ]:
import sys
sys.path.insert(0, '..')

import types
import torch
import torch.nn as nn

from disease_state import (
    ArchitectureConfig, QWEN_3B, QWEN_7B, QWEN_32B,
    arch_from_model_config, get_active_arch, set_active_arch,
    BraakStage, BrainRegion, DamagePhase, DamageIntensity,
    DamageConfig, DamageTarget,
    NoiseConfig, PruningConfig, ConnectivityConfig,
    CompensationConfig, LayerDamageState,
    DiseaseState, layer_to_brain_region,
    SparseDeltaT, WeightSnapshot,
)

# ---------------------------------------------------------------------------
# Test runner
# ---------------------------------------------------------------------------
_PASS = _FAIL = 0
_failures = []

def check(name: str, condition: bool, detail: str = '') -> bool:
    global _PASS, _FAIL
    if condition:
        _PASS += 1
        print(f'  PASS  {name}')
    else:
        _FAIL += 1
        msg = f'  FAIL  {name}' + (f'  →  {detail}' if detail else '')
        print(msg)
        _failures.append(msg)
    return condition

def section(title: str):
    print(f'\n{"="*64}')
    print(f'  {title}')
    print(f'{"="*64}')

def final_summary():
    total = _PASS + _FAIL
    print(f'\n{"="*64}')
    print(f'  Results: {_PASS}/{total} passed')
    if _FAIL:
        print(f'  {_FAIL} FAILURE(S):')
        for f in _failures:
            print(f'    {f}')
    else:
        print('  ALL TESTS PASSED')
    print(f'{"="*64}')
    assert _FAIL == 0, f'{_FAIL} test(s) failed — see output above'

# ---------------------------------------------------------------------------
# Mock Qwen model — matches real model attribute paths used by damage functions
# ---------------------------------------------------------------------------
class _MA(nn.Module):   # MockAttention
    def __init__(self, d):
        super().__init__()
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, d, bias=False)
        self.v_proj = nn.Linear(d, d, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)

class _MM(nn.Module):   # MockMLP
    def __init__(self, d, ffn_d):
        super().__init__()
        self.gate_proj = nn.Linear(d, ffn_d, bias=False)
        self.up_proj   = nn.Linear(d, ffn_d, bias=False)
        self.down_proj = nn.Linear(ffn_d, d, bias=False)

class _ML(nn.Module):   # MockLayer
    def __init__(self, d, ffn_d):
        super().__init__()
        self.self_attn = _MA(d)
        self.mlp = _MM(d, ffn_d)
        self.input_layernorm = nn.LayerNorm(d)
        self.post_attention_layernorm = nn.LayerNorm(d)

class _MQI(nn.Module):  # MockQwenInner  (== model.model)
    def __init__(self, n, d, ffn_d):
        super().__init__()
        self.embed_tokens = nn.Embedding(64, d)
        self.layers = nn.ModuleList([_ML(d, ffn_d) for _ in range(n)])

class MockQwen(nn.Module):
    """6-layer, d=16, ffn=32 mock — CPU FP32. Attribute paths match Qwen."""
    def __init__(self, n=6, d=16, ffn_d=32):
        super().__init__()
        self.model = _MQI(n, d, ffn_d)
        self.lm_head = nn.Linear(d, 64, bias=False)

def fresh_mock():
    """Return a new MockQwen with QWEN_3B active (6-layer → BraakStage.I_II)."""
    set_active_arch(QWEN_3B)
    return MockQwen(n=6, d=16, ffn_d=32)

def orig_weights(model):
    return {n: p.data.clone() for n, p in model.named_parameters()}

print('Setup complete. Mock model defined.')

## Section 1 — `disease_state`

In [ ]:
# ── 1a. ArchitectureConfig ──────────────────────────────────────────────────
section('1a  ArchitectureConfig — preset constants and gqa_ratio')

check('QWEN_3B.num_layers == 36',        QWEN_3B.num_layers == 36)
check('QWEN_3B.hidden_dim == 2048',      QWEN_3B.hidden_dim == 2048)
check('QWEN_3B.num_q_heads == 16',       QWEN_3B.num_q_heads == 16)
check('QWEN_3B.num_kv_heads == 2',       QWEN_3B.num_kv_heads == 2)
check('QWEN_3B.gqa_ratio == 8',          QWEN_3B.gqa_ratio == 8)

check('QWEN_7B.num_layers == 28',        QWEN_7B.num_layers == 28)
check('QWEN_7B.gqa_ratio == 7',          QWEN_7B.gqa_ratio == 7)

check('QWEN_32B.num_layers == 64',       QWEN_32B.num_layers == 64)
check('QWEN_32B.gqa_ratio == 5',         QWEN_32B.gqa_ratio == 5)

# ── 1b. arch_from_model_config ─────────────────────────────────────────────
section('1b  arch_from_model_config — reads from mock HuggingFace config')

mock_hf_cfg = types.SimpleNamespace(
    num_hidden_layers=36, hidden_size=2048, intermediate_size=11008,
    num_attention_heads=16, num_key_value_heads=2, vocab_size=151936,
    _name_or_path='test-mock-3b',
)
a = arch_from_model_config(mock_hf_cfg)
check('num_layers read from config',          a.num_layers      == mock_hf_cfg.num_hidden_layers)
check('hidden_dim read from config',          a.hidden_dim      == mock_hf_cfg.hidden_size)
check('ffn_intermediate read from config',    a.ffn_intermediate == mock_hf_cfg.intermediate_size)
check('num_q_heads read from config',         a.num_q_heads     == mock_hf_cfg.num_attention_heads)
check('num_kv_heads read from config',        a.num_kv_heads    == mock_hf_cfg.num_key_value_heads)
check('vocab_size read from config',          a.vocab_size      == mock_hf_cfg.vocab_size)
check('name taken from _name_or_path',        a.name            == 'test-mock-3b')
check('gqa_ratio derived correctly',          a.gqa_ratio       == 16 // 2)

# _name_or_path missing → falls back to model_type
mock_hf_cfg2 = types.SimpleNamespace(
    num_hidden_layers=28, hidden_size=3584, intermediate_size=18944,
    num_attention_heads=28, num_key_value_heads=4, vocab_size=151936,
    model_type='qwen2',
)
a2 = arch_from_model_config(mock_hf_cfg2)
check('name falls back to model_type when _name_or_path missing', a2.name == 'qwen2')
check('fallback arch gqa_ratio == 7',         a2.gqa_ratio == 7)

In [ ]:
# ── 1c. BraakStage ─────────────────────────────────────────────────────────
section('1c  BraakStage — layer_bounds, severity_cap, ordinal, scaling')

set_active_arch(QWEN_3B)   # 36 layers

s, e = BraakStage.I_II.layer_bounds
check('I_II: starts at 0',                   s == 0)
check('I_II: ends at layer 5 (ref 36-layer)', e == 5)
check('I_II: layer_range length == 6',        len(BraakStage.I_II.layer_range) == 6)

s, e = BraakStage.VI.layer_bounds
check('VI: starts at 0',                      s == 0)
check('VI: ends at last layer (35)',           e == 35)
check('VI: layer_range covers all 36 layers', len(BraakStage.VI.layer_range) == 36)

check('I_II severity_cap == 0.3',  BraakStage.I_II.severity_cap  == 0.3)
check('III_IV severity_cap == 0.5',BraakStage.III_IV.severity_cap == 0.5)
check('V severity_cap == 0.7',     BraakStage.V.severity_cap      == 0.7)
check('VI severity_cap == 1.0',    BraakStage.VI.severity_cap      == 1.0)

check('ordinals are 1-2-3-4', [
    BraakStage.I_II.ordinal, BraakStage.III_IV.ordinal,
    BraakStage.V.ordinal, BraakStage.VI.ordinal,
] == [1, 2, 3, 4])

# Monotonic: each stage's end >= previous stage's end
ends = [BraakStage.I_II.layer_bounds[1], BraakStage.III_IV.layer_bounds[1],
        BraakStage.V.layer_bounds[1], BraakStage.VI.layer_bounds[1]]
check('stage ends are strictly increasing', ends == sorted(ends) and len(set(ends)) == 4)

# Scales with arch
set_active_arch(QWEN_7B)   # 28 layers
_, e7 = BraakStage.VI.layer_bounds
check('QWEN_7B: VI ends at layer 27', e7 == 27)

set_active_arch(QWEN_32B)  # 64 layers
_, e64 = BraakStage.VI.layer_bounds
check('QWEN_32B: VI ends at layer 63', e64 == 63)

set_active_arch(QWEN_3B)   # restore for remaining tests

In [ ]:
# ── 1d. BrainRegion ─────────────────────────────────────────────────────────
section('1d  BrainRegion — partition, vulnerability, layer_to_brain_region')

set_active_arch(QWEN_3B)

# All regions must partition [0, 35] exactly — no gaps, no overlaps
covered = []
for region in BrainRegion:
    covered.extend(region.layer_range)
check('regions cover all 36 layers (QWEN_3B)', len(covered) == 36)
check('no gaps or overlaps (QWEN_3B)',          sorted(covered) == list(range(36)))

# Vulnerability order: ENTORHINAL first, PREFRONTAL last
check('ENTORHINAL most vulnerable (1)',   BrainRegion.ENTORHINAL.ad_vulnerability_order == 1)
check('HIPPOCAMPAL second (2)',           BrainRegion.HIPPOCAMPAL.ad_vulnerability_order == 2)
check('TEMPORAL_ASSOCIATION third (3)',   BrainRegion.TEMPORAL_ASSOCIATION.ad_vulnerability_order == 3)
check('PREFRONTAL least vulnerable (4)', BrainRegion.PREFRONTAL.ad_vulnerability_order == 4)

# layer_to_brain_region
check('layer 0 → ENTORHINAL',  layer_to_brain_region(0)  == BrainRegion.ENTORHINAL)
check('layer 35 → PREFRONTAL', layer_to_brain_region(35) == BrainRegion.PREFRONTAL)

# Contiguous: HIPPOCAMPAL starts immediately after ENTORHINAL ends
ec = BrainRegion.ENTORHINAL.layer_range
hc = BrainRegion.HIPPOCAMPAL.layer_range
check('HIPPOCAMPAL starts immediately after ENTORHINAL', hc.start == ec.stop)

# Out-of-range raises ValueError
try:
    layer_to_brain_region(36)
    check('layer 36 raises ValueError', False)
except ValueError:
    check('layer 36 raises ValueError', True)

try:
    layer_to_brain_region(-1)
    check('layer -1 raises ValueError', False)
except ValueError:
    check('layer -1 raises ValueError', True)

# Scales correctly with QWEN_7B (28 layers)
set_active_arch(QWEN_7B)
covered7 = []
for region in BrainRegion:
    covered7.extend(region.layer_range)
check('regions cover all 28 layers (QWEN_7B)', sorted(covered7) == list(range(28)))

set_active_arch(QWEN_3B)

In [ ]:
# ── 1e. DamageIntensity ──────────────────────────────────────────────────────
section('1e  DamageIntensity — ranges and midpoints')

for intensity in DamageIntensity:
    lo, hi = intensity.noise_range
    mp = intensity.midpoint_noise
    check(f'{intensity.value}: midpoint_noise in [lo,hi]',
          lo <= mp <= hi, f'lo={lo} mp={mp} hi={hi}')
    lo, hi = intensity.pruning_range
    mp = intensity.midpoint_prune
    check(f'{intensity.value}: midpoint_prune in [lo,hi]',
          lo <= mp <= hi, f'lo={lo} mp={mp} hi={hi}')
    lo, hi = intensity.neuron_kill_range
    mp = intensity.midpoint_kill
    check(f'{intensity.value}: midpoint_kill in [lo,hi]',
          lo <= mp <= hi, f'lo={lo} mp={mp} hi={hi}')

# Strictly increasing noise midpoints across tiers
mps = [i.midpoint_noise for i in [
    DamageIntensity.SUBCLINICAL, DamageIntensity.MILD,
    DamageIntensity.MODERATE,   DamageIntensity.SEVERE]]
check('noise midpoints strictly increasing across tiers', mps == sorted(mps) and len(set(mps)) == 4)

# ── 1f. DamageConfig — effective_* clamping ──────────────────────────────────
section('1f  DamageConfig — effective_* clamping by severity_cap')

set_active_arch(QWEN_3B)

# Stage I_II cap = 0.3; SEVERE midpoints are all > 0.3
cfg = DamageConfig(
    braak_stage=BraakStage.I_II,
    damage_phase=DamagePhase.AMYLOID,
    intensity=DamageIntensity.SEVERE,
)
check('effective_noise_std clamped to 0.3',  cfg.effective_noise_std()  == 0.3)
check('effective_prune_rate clamped to 0.3', cfg.effective_prune_rate() == 0.3)
check('effective_kill_rate clamped to 0.3',  cfg.effective_kill_rate()  == 0.3)

# Override below cap passes through unchanged
cfg.noise_std_override = 0.05
check('noise_std_override (0.05) below cap passes through', cfg.effective_noise_std() == 0.05)

# Override above cap is still clamped
cfg.noise_std_override = 0.99
check('noise_std_override (0.99) above cap is clamped', cfg.effective_noise_std() == 0.3)

# Stage VI has no cap — SEVERE damage is not clamped
cfg_vi = DamageConfig(
    braak_stage=BraakStage.VI,
    damage_phase=DamagePhase.TAU,
    intensity=DamageIntensity.SEVERE,
)
check('Stage VI: noise above 0.3 is NOT clamped',
      cfg_vi.effective_noise_std() > 0.3)

In [ ]:
# ── 1g. CompensationConfig ───────────────────────────────────────────────────
section('1g  CompensationConfig — depletion, floor, effective_factor')

comp = CompensationConfig(
    compensation_reserve=1.0,
    base_compensation_factor=0.2,
    depletion_per_increment=0.1,
    min_reserve=0.0,
)
check('initial effective_factor == 0.2',     abs(comp.effective_compensation_factor() - 0.2) < 1e-9)
check('not exhausted initially',             not comp.is_exhausted)

comp.deplete(1)
check('reserve after 1 deplete == 0.9',      abs(comp.compensation_reserve - 0.9) < 1e-9)
check('effective_factor after 1 deplete',    abs(comp.effective_compensation_factor() - 0.18) < 1e-9)

comp.deplete(100)   # way past floor
check('reserve floors at min_reserve (0.0)', comp.compensation_reserve == 0.0)
check('is_exhausted == True at floor',       comp.is_exhausted)
check('effective_factor == 0 when exhausted', comp.effective_compensation_factor() == 0.0)

# ── 1h. LayerDamageState ─────────────────────────────────────────────────────
section('1h  LayerDamageState — total_damage, GQA coupling, invalidate_masks')

set_active_arch(QWEN_3B)

ls = LayerDamageState(layer_idx=0, brain_region=BrainRegion.ENTORHINAL)
check('initial total_damage == 0',       ls.total_damage == 0.0)
check('not severely damaged initially',  not ls.is_severely_damaged)
check('dead_neuron_count == 0',          ls.dead_neuron_count == 0)
check('ablated_kv_head_count == 0',      ls.ablated_kv_head_count == 0)
check('affected_q_head_count == 0',      ls.affected_q_head_count == 0)

# total_damage weighted formula: 0.25n + 0.25p + 0.35d + 0.15c
ls.noise_level = 1.0; ls.prune_level = 1.0
ls.neuron_death_level = 1.0; ls.connectivity_disruption = 1.0
check('total_damage == 1.0 at max',      abs(ls.total_damage - 1.0) < 1e-9)
check('is_severely_damaged at total==1', ls.is_severely_damaged)

ls2 = LayerDamageState(layer_idx=3, brain_region=BrainRegion.HIPPOCAMPAL)
ls2.ablated_kv_heads = [0, 1, 2]
check('ablated_kv_head_count == 3',                        ls2.ablated_kv_head_count == 3)
check('affected_q_head_count == 3 * gqa_ratio (24)',       ls2.affected_q_head_count == 3 * QWEN_3B.gqa_ratio)

# invalidate_masks clears cache
ls2.pruning_mask_indices = {'ffn_gate': [0, 1, 2], 'ffn_up': [5, 6]}
ls2.masks_computed = True
ls2.invalidate_masks()
check('invalidate_masks: pruning_mask_indices cleared', ls2.pruning_mask_indices == {})
check('invalidate_masks: masks_computed set False',     not ls2.masks_computed)

In [ ]:
# ── 1i. DiseaseState ─────────────────────────────────────────────────────────
section('1i  DiseaseState — factory methods, layer_states, record_damage_increment')

set_active_arch(QWEN_3B)

# healthy()
s = DiseaseState.healthy()
check('healthy: braak == I_II',              s.braak_stage == BraakStage.I_II)
check('healthy: phase == AMYLOID',           s.damage_phase == DamagePhase.AMYLOID)
check('healthy: intensity == SUBCLINICAL',   s.intensity == DamageIntensity.SUBCLINICAL)
check('healthy: epoch == 0',                 s.epoch == 0)
check('healthy: total_damage_accumulated == 0', s.total_damage_accumulated == 0.0)
check('healthy: not is_phase_two',           not s.is_phase_two)
check('healthy: 36 layer_states (QWEN_3B)',  len(s.layer_states) == 36)
check('healthy: all layer indices present',  set(s.layer_states.keys()) == set(range(36)))
for i, ls in s.layer_states.items():
    if not check(f'healthy: layer {i} total_damage == 0', ls.total_damage == 0.0):
        break

# from_braak_stage
s2 = DiseaseState.from_braak_stage(BraakStage.III_IV, DamagePhase.TAU, DamageIntensity.MODERATE)
check('from_braak_stage: braak == III_IV',   s2.braak_stage == BraakStage.III_IV)
check('from_braak_stage: phase == TAU',      s2.damage_phase == DamagePhase.TAU)
check('from_braak_stage: is_phase_two',      s2.is_phase_two)

# record_damage_increment
s3 = DiseaseState.healthy()
s3.layer_states[0].noise_level = 0.5
s3.layer_states[0].masks_computed = True
s3.record_damage_increment()
check('after record: epoch == 1',            s3.epoch == 1)
check('after record: compensation depleted', s3.compensation.compensation_reserve < 1.0)
check('after record: total_damage_accumulated > 0', s3.total_damage_accumulated > 0.0)
check('after record: masks invalidated (layer 0)', not s3.layer_states[0].masks_computed)

# mean_damage_in_region
s4 = DiseaseState.healthy()
entorhinal_layers = list(BrainRegion.ENTORHINAL.layer_range)
for i in entorhinal_layers:
    s4.layer_states[i].noise_level = 0.4
mean = s4.mean_damage_in_region(BrainRegion.ENTORHINAL)
check('mean_damage_in_region > 0 after noise',      mean > 0.0)
check('mean_damage_in_region == 0 for PREFRONTAL', s4.mean_damage_in_region(BrainRegion.PREFRONTAL) == 0.0)

# severely_damaged_layers
s5 = DiseaseState.healthy()
s5.layer_states[2].noise_level = 1.0
s5.layer_states[2].prune_level = 1.0
severe = s5.severely_damaged_layers()
check('severely_damaged_layers includes layer 2', 2 in severe)
check('layer 0 not in severely_damaged_layers',   0 not in severe)

# summary() returns expected keys
summ = s.summary()
for key in ['braak_stage', 'damage_phase', 'intensity', 'epoch',
            'compensation_reserve', 'total_damage_accumulated', 'region_damage']:
    check(f'summary has key: {key}', key in summ)

# layer_states scales with active arch
set_active_arch(QWEN_7B)
s7 = DiseaseState.healthy()
check('QWEN_7B: 28 layer_states',  len(s7.layer_states) == 28)
set_active_arch(QWEN_3B)

## Section 2 — `damage/noise`

In [ ]:
from damage.noise import inject_noise, restore_noise

section('2a  inject_noise — targeted weights modified, untargeted untouched')

mock = fresh_mock()   # 6-layer, QWEN_3B active
orig = orig_weights(mock)

state = DiseaseState.from_braak_stage(BraakStage.I_II, DamagePhase.AMYLOID, DamageIntensity.MILD)
state.damage_config.noise_std_override       = 0.1
state.damage_config.noise.apply_to_attention = True
state.damage_config.noise.apply_to_ffn       = True
state.damage_config.noise.apply_to_embeddings = True
state.damage_config.noise.apply_to_layer_norm = False
state.damage_config.noise.seed = 42

ATTN = {'q_proj', 'k_proj', 'v_proj', 'o_proj'}
FFN  = {'gate_proj', 'up_proj', 'down_proj'}

with torch.no_grad():
    snapshot = inject_noise(mock, state)

for name, p in mock.named_parameters():
    changed = not torch.equal(p.data, orig[name])
    parts = name.split('.')
    leaf = parts[-2] if len(parts) >= 2 else ''
    if leaf in ATTN or leaf in FFN or 'embed_tokens' in name:
        check(f'modified: {name}', changed)
    if 'layernorm' in name.lower() or name.startswith('lm_head'):
        check(f'untouched: {name}', not changed)

section('2b  inject_noise — depth scaling (layer 0 noise > last layer noise)')

delta0 = (mock.model.layers[0].mlp.gate_proj.weight.data
          - orig['model.layers.0.mlp.gate_proj.weight']).abs().mean().item()
delta5 = (mock.model.layers[5].mlp.gate_proj.weight.data
          - orig['model.layers.5.mlp.gate_proj.weight']).abs().mean().item()
check('layer 0 noise > layer 5 noise',   delta0 > delta5,
      f'layer 0: {delta0:.5f}, layer 5: {delta5:.5f}')
check('depth ratio ≈ 2.0×',              1.8 < delta0 / delta5 < 2.2,
      f'ratio = {delta0/delta5:.2f}')

section('2c  inject_noise — LayerDamageState.noise_level updated')

for i in state.affected_layers:
    check(f'noise_level > 0: layer {i}', state.layer_state(i).noise_level > 0)

section('2d  restore_noise — FP32 round-trip (atol 1e-6)')

with torch.no_grad():
    restore_noise(mock, snapshot)

max_err = 0.0
for name, p in mock.named_parameters():
    err = (p.data - orig[name]).abs().max().item()
    if err > max_err:
        max_err = err
    check(f'restored: {name}', err < 1e-6, f'max|err|={err:.2e}')
print(f'  Overall max|err| = {max_err:.2e}')

section('2e  sanity_check_noise() — saves/restores active arch')

set_active_arch(QWEN_7B)   # set to non-default to verify it gets restored
from damage.noise import sanity_check_noise
result = sanity_check_noise(verbose=False)
check('sanity_check_noise returns True', result)
check('active arch restored to QWEN_7B after sanity check',
      get_active_arch().num_layers == QWEN_7B.num_layers)
set_active_arch(QWEN_3B)

## Section 3 — `damage/pruning`

In [ ]:
from damage.pruning import compute_pruning_masks, prune_weights, restore_pruning

PRUNE_RATE = 0.10

section('3a  compute_pruning_masks — index counts correct')

mock = fresh_mock()
orig = orig_weights(mock)

state = DiseaseState.from_braak_stage(BraakStage.I_II, DamagePhase.TAU, DamageIntensity.MILD,
    compensation=CompensationConfig(compensation_reserve=1.0, base_compensation_factor=0.2))
state.damage_config.prune_rate_override = PRUNE_RATE
state.damage_config.pruning.prune_type = 'magnitude'
state.damage_config.pruning.seed = 0

with torch.no_grad():
    compute_pruning_masks(mock, state)

for layer_idx in state.affected_layers:
    ls = state.layer_state(layer_idx)
    check(f'masks_computed=True: layer {layer_idx}', ls.masks_computed)
    for target in state.damage_config.pruning.target_components:
        from damage.pruning import _get_param_for_target
        param = _get_param_for_target(mock.model.layers[layer_idx], target)
        if param is None:
            continue
        expected_k = int(PRUNE_RATE * param.numel())
        got_k = len(ls.pruning_mask_indices.get(target.value, []))
        check(f'mask count layer {layer_idx} {target.value}: {expected_k}',
              got_k == expected_k, f'got {got_k}')

section('3b  prune_weights — magnitude ordering and exact zero count')

eff_factor = state.compensation.effective_compensation_factor()
expected_comp_mult = 1.0 + eff_factor * PRUNE_RATE

with torch.no_grad():
    snapshot = prune_weights(mock, state)

# Magnitude ordering on layer 0 gate_proj
gate_orig = orig['model.layers.0.mlp.gate_proj.weight']
gate_now  = mock.model.layers[0].mlp.gate_proj.weight.data
gate_mask = gate_now.view(-1) == 0
n_zeroed  = gate_mask.sum().item()
n_expected = int(PRUNE_RATE * gate_now.numel())
check('exact zero count in gate_proj layer 0', n_zeroed == n_expected,
      f'got {n_zeroed}, expected {n_expected}')

orig_flat = gate_orig.view(-1)
pruned_mags   = orig_flat[gate_mask].abs()
survivor_mags = orig_flat[~gate_mask].abs()
check('all pruned |w| ≤ min survivor |w|',
      pruned_mags.max().item() <= survivor_mags.min().item() + 1e-7,
      f'max_pruned={pruned_mags.max().item():.5f} min_surv={survivor_mags.min().item():.5f}')

section('3c  prune_weights — compensation multiplier applied to survivors')

gate_after = mock.model.layers[0].mlp.gate_proj.weight.data.view(-1)
alive = gate_after != 0
if alive.any():
    ratios = gate_after[alive] / gate_orig.view(-1)[alive]
    mean_ratio = ratios.mean().item()
    check('mean survivor ratio ≈ comp_mult',
          abs(mean_ratio - expected_comp_mult) < 1e-5,
          f'got {mean_ratio:.6f}, expected {expected_comp_mult:.6f}')

section('3d  prune_weights — prune_level updated in LayerDamageState')

for i in state.affected_layers:
    check(f'prune_level > 0: layer {i}', state.layer_state(i).prune_level > 0)

section('3e  restore_pruning — FP32 round-trip (atol 1e-5)')

with torch.no_grad():
    restore_pruning(mock, snapshot)

max_err = 0.0
for name, p in mock.named_parameters():
    err = (p.data - orig[name]).abs().max().item()
    if err > max_err:
        max_err = err
    check(f'restored: {name}', err < 1e-5, f'max|err|={err:.2e}')
print(f'  Overall max|err| = {max_err:.2e}')

section('3f  sanity_check_pruning() — saves/restores active arch')

set_active_arch(QWEN_7B)
from damage.pruning import sanity_check_pruning
result = sanity_check_pruning(verbose=False)
check('sanity_check_pruning returns True', result)
check('active arch restored to QWEN_7B after sanity check',
      get_active_arch().num_layers == QWEN_7B.num_layers)
set_active_arch(QWEN_3B)

## Section 4 — `damage/mechanisms`

In [ ]:
from damage.mechanisms import apply_all_damage
from qwen_hooks import _restore_static_damage

# ── 4a. Noise only (prune_rate=0) ────────────────────────────────────────────
section('4a  apply_all_damage — noise only (prune_rate=0)')

mock = fresh_mock()
orig = orig_weights(mock)

state = DiseaseState.from_braak_stage(BraakStage.I_II, DamagePhase.AMYLOID, DamageIntensity.MILD)
state.damage_config.noise_std_override  = 0.05
state.damage_config.prune_rate_override = 0.0
state.damage_config.noise.apply_to_layer_norm = False
state.damage_config.noise.seed = 1

with torch.no_grad():
    snapshot = apply_all_damage(mock, state)

check('snapshot is a list',                     isinstance(snapshot, list))
check('snapshot non-empty',                     len(snapshot) > 0)
check('all entries are dicts',                  all(isinstance(d, dict) for d in snapshot))
check('all entries have delta_type',            all('delta_type' in d for d in snapshot))
check('all entries have param_name',            all('param_name' in d for d in snapshot))
check('all delta_types are additive_noise',     all(d['delta_type'] == 'additive_noise' for d in snapshot))

n_changed = sum(1 for n, p in mock.named_parameters() if not torch.equal(p.data, orig[n]))
check('weights were modified',                  n_changed > 0, f'{n_changed} params changed')

# No pruning deltas when prune_rate=0
pruning_deltas = [d for d in snapshot if d['delta_type'] == 'sparse_zero_with_compensation']
check('no pruning deltas when prune_rate=0',   len(pruning_deltas) == 0)

with torch.no_grad():
    _restore_static_damage(mock, snapshot)
for name, p in mock.named_parameters():
    err = (p.data - orig[name]).abs().max().item()
    check(f'noise-only restore: {name.split(".",2)[-1]}', err < 1e-6, f'err={err:.2e}')

# ── 4b. Noise + pruning combined ─────────────────────────────────────────────
section('4b  apply_all_damage — noise + pruning combined (reversed restore order)')

mock = fresh_mock()
orig = orig_weights(mock)

state = DiseaseState.from_braak_stage(BraakStage.I_II, DamagePhase.TAU, DamageIntensity.MILD)
state.damage_config.noise_std_override  = 0.05
state.damage_config.prune_rate_override = 0.10
state.damage_config.noise.apply_to_layer_norm = False
state.damage_config.noise.seed = 2
state.damage_config.pruning.seed = 2

with torch.no_grad():
    snapshot = apply_all_damage(mock, state)

noise_deltas   = [d for d in snapshot if d['delta_type'] == 'additive_noise']
pruning_deltas = [d for d in snapshot if d['delta_type'] == 'sparse_zero_with_compensation']
check('snapshot contains noise deltas',   len(noise_deltas) > 0)
check('snapshot contains pruning deltas', len(pruning_deltas) > 0)

# Noise deltas come first (applied first), pruning deltas come after
noise_positions   = [i for i, d in enumerate(snapshot) if d['delta_type'] == 'additive_noise']
pruning_positions = [i for i, d in enumerate(snapshot) if d['delta_type'] == 'sparse_zero_with_compensation']
check('noise deltas precede pruning deltas in snapshot',
      max(noise_positions) < min(pruning_positions))

# Restoration must correctly undo both (undo-stack semantics via reversed(snapshot))
with torch.no_grad():
    _restore_static_damage(mock, snapshot)

max_err = 0.0
for name, p in mock.named_parameters():
    err = (p.data - orig[name]).abs().max().item()
    if err > max_err:
        max_err = err
    check(f'noise+prune restore: {name.split(".",2)[-1]}', err < 1e-5, f'err={err:.2e}')
print(f'  max|err| across all params = {max_err:.2e}')

## Section 5 — `qwen_hooks.damage_context`

In [ ]:
from qwen_hooks import damage_context

def _noise_state(std=0.05, seed=7):
    """Create a noise-only DiseaseState for damage_context tests."""
    s = DiseaseState.from_braak_stage(BraakStage.I_II, DamagePhase.AMYLOID, DamageIntensity.SUBCLINICAL)
    s.damage_config.noise_std_override  = std
    s.damage_config.prune_rate_override = 0.0
    s.damage_config.noise.apply_to_layer_norm = False
    s.damage_config.noise.seed = seed
    s.damage_config.connectivity.kv_cache_corruption_rate = 0.0  # no hooks on mock
    return s

# ── 5a. Basic round-trip ─────────────────────────────────────────────────────
section('5a  damage_context — weights modified inside, restored outside')

mock = fresh_mock()
orig = orig_weights(mock)

n_changed = 0
with damage_context(mock, _noise_state()):
    for name, p in mock.named_parameters():
        if not torch.equal(p.data, orig[name]):
            n_changed += 1

check('weights changed inside context', n_changed > 0, f'{n_changed} params')

for name, p in mock.named_parameters():
    err = (p.data - orig[name]).abs().max().item()
    check(f'restored after exit: {name.split(".",2)[-1]}', err < 1e-6, f'err={err:.2e}')

# ── 5b. Exception safety ─────────────────────────────────────────────────────
section('5b  damage_context — weights restored even when body raises')

mock = fresh_mock()
orig = orig_weights(mock)

raised = False
try:
    with damage_context(mock, _noise_state(seed=99)):
        raise ValueError('intentional test exception')
except ValueError:
    raised = True

check('exception propagated correctly', raised)

for name, p in mock.named_parameters():
    err = (p.data - orig[name]).abs().max().item()
    check(f'exception safety — restored: {name.split(".",2)[-1]}', err < 1e-6, f'err={err:.2e}')

# ── 5c. No hooks registered when KV rate is 0 ────────────────────────────────
section('5c  damage_context — no forward hooks when kv_cache_corruption_rate=0')

mock = fresh_mock()

# Count hooks before
def count_hooks(m):
    return sum(len(mod._forward_hooks) for mod in m.modules())

hooks_before = count_hooks(mock)
with damage_context(mock, _noise_state()):
    hooks_during = count_hooks(mock)
hooks_after = count_hooks(mock)

check('no extra hooks during context (kv_rate=0)', hooks_during == hooks_before)
check('no hooks leaked after context exit',        hooks_after  == hooks_before)

# ── 5d. masks_computed set and then invalidated by record_damage_increment ───
section('5d  damage_context — masks computed on entry, invalidated by record_damage_increment')

mock = fresh_mock()
state = DiseaseState.from_braak_stage(BraakStage.I_II, DamagePhase.TAU, DamageIntensity.MILD)
state.damage_config.noise_std_override  = 0.01
state.damage_config.prune_rate_override = 0.05
state.damage_config.noise.seed = 0
state.damage_config.pruning.seed = 0
state.damage_config.connectivity.kv_cache_corruption_rate = 0.0

with damage_context(mock, state):
    masks_during = {i: state.layer_states[i].masks_computed for i in state.affected_layers}

check('masks computed for all affected layers inside context',
      all(masks_during.values()))

state.record_damage_increment()
check('masks invalidated after record_damage_increment',
      all(not state.layer_states[i].masks_computed for i in state.affected_layers))

## Section 6 — Real model integration

Loads `Qwen/Qwen2.5-3B-Instruct` and runs an end-to-end test on the full BF16 model. Requires ~8 GB VRAM (or CPU RAM if no GPU).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
print(f'Loading {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
real_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
real_model.eval()

real_arch = arch_from_model_config(real_model.config)
set_active_arch(real_arch)
print(f'Loaded. arch: {real_arch.name} — {real_arch.num_layers} layers, '
      f'gqa_ratio={real_arch.gqa_ratio}')

In [ ]:
cfg = real_model.config
n   = cfg.num_hidden_layers

section('6a  arch_from_model_config matches model.config exactly')
check('num_layers',       real_arch.num_layers      == cfg.num_hidden_layers)
check('hidden_dim',       real_arch.hidden_dim      == cfg.hidden_size)
check('ffn_intermediate', real_arch.ffn_intermediate == cfg.intermediate_size)
check('num_q_heads',      real_arch.num_q_heads     == cfg.num_attention_heads)
check('num_kv_heads',     real_arch.num_kv_heads    == cfg.num_key_value_heads)
check('vocab_size',       real_arch.vocab_size      == cfg.vocab_size)

section('6b  BrainRegion partitions all model layers')
covered = []
for region in BrainRegion:
    covered.extend(region.layer_range)
check(f'regions cover all {n} layers', sorted(covered) == list(range(n)))

section('6c  DiseaseState.healthy() has num_hidden_layers layer states')
ds = DiseaseState.healthy()
check(f'len(layer_states) == {n}', len(ds.layer_states) == n)

section('6d  damage_context round-trip on full BF16 model')
pre = {name: p.data.clone() for name, p in real_model.named_parameters()}

rstate = DiseaseState.from_braak_stage(
    BraakStage.I_II, DamagePhase.AMYLOID, DamageIntensity.SUBCLINICAL)
rstate.damage_config.noise_std_override  = 0.01
rstate.damage_config.prune_rate_override = 0.0
rstate.damage_config.noise.apply_to_layer_norm = False
rstate.damage_config.noise.seed = 42
rstate.damage_config.connectivity.kv_cache_corruption_rate = 0.0

n_changed = 0
with damage_context(real_model, rstate):
    for name, p in real_model.named_parameters():
        if not torch.equal(p.data, pre[name]):
            n_changed += 1

check(f'weights changed inside context ({n_changed} params)', n_changed > 0)

max_err = 0.0
worst   = ''
for name, p in real_model.named_parameters():
    err = (p.data.float() - pre[name].float()).abs().max().item()
    if err > max_err:
        max_err = err; worst = name
    assert err < 1e-2, f'restore err too large for {name}: {err:.2e}'
check(f'all {len(pre)} params restored (BF16 tol 1e-2)', max_err < 1e-2,
      f'worst err={max_err:.2e} ({worst})')

section('6e  exception safety on real model')
pre2 = {name: p.data.clone() for name, p in real_model.named_parameters()}
try:
    with damage_context(real_model, rstate):
        raise RuntimeError('intentional')
except RuntimeError:
    pass
max_err2 = max(
    (p.data.float() - pre2[name].float()).abs().max().item()
    for name, p in real_model.named_parameters()
)
check('exception safety — real model restored', max_err2 < 1e-2, f'max_err={max_err2:.2e}')

section('6f  Undamaged baseline generates coherent output')
prompt = 'The capital of France is'
inputs = tokenizer(prompt, return_tensors='pt').to(real_model.device)
with torch.no_grad():
    out = real_model.generate(**inputs, max_new_tokens=15, do_sample=False)
response = tokenizer.decode(out[0], skip_special_tokens=True)
print(f'  {response}')
check('generates more tokens than input', len(out[0]) > inputs['input_ids'].shape[1])

In [ ]:
final_summary()